# LELA60331 Week 11 Seminar: Training and evaluating a softmax classifier

Last week we built all the parts of a **softmax classifier** for three classes: a score for each class, the softmax function, the predicted class and the cross-entropy loss. We used weights that had already been trained. This week we **train** a softmax classifier ourselves, and evaluate it. As for logistic regression in Weeks 5 and 7, training works by gradient descent, and we build it up one step at a time:

| Part | What you do |
|---|---|
| 0 | Run the set-up cells: everything you built last week |
| 1 | Work out the **errors** of the predictions for one text |
| 2 | Work out the **gradients** for one text |
| 3 | Make **one** improvement to the weights |
| 4 | Do the same for all the texts at once |
| 5 | Repeat the improvement many times: **training**, with the parts from LELA60341 Week 8 |
| 6 | Use the trained model |
| 7 | Evaluate it with **precision and recall** for each class |

Each activity has a cell to complete (replace every `???`) followed by a ✅ **check** cell.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used after each activity.
# You don't need to understand how it works: it prints ✅ when your answer is right and ❌ when it isn't yet.
def check(label, condition):
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Where we got to last week

The next cells set up everything from last week, so that this sheet works on its own:

| Name | What it is | Week 10 |
|---|---|---|
| `dot`, `mat_vec` | the dot product, and matrix-vector multiplication | Weeks 4 and 7 |
| `X`, `labels`, `class_names` | the 60 simulated texts (one row of features each), their classes, and the class names | Part 1 |
| `one_hot(c, n_classes)`, `Y` | one-hot vectors, and the one-hot vector of every text | Part 2 |
| `scores(W, b, x)` | the score of each class for the features `x` | Part 3 |
| `softmax(z)` | turns the scores into probabilities | Part 4 |
| `argmax(values)` | the position of the largest value | Part 5 |
| `loss_one(q, c)`, `average_loss(W, b, X, labels)` | the cross-entropy loss | Part 6 |
| `Parameter`, `Module`, `Linear`, `SGD` | the parts from the LELA60341 Week 8 workbook | Part 7 |

Run them.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def dot(a, b):
    # the dot product of two lists of the same length (Week 4)
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # matrix-vector multiplication: the dot product of each row of the matrix with the vector (Week 7)
    result = []
    for row in matrix:
        result.append(dot(row, vector))
    return result

def one_hot(c, n_classes):
    vector = [0] * n_classes
    vector[c] = 1
    return vector

def scores(W, b, x):
    # one score per class: the dot product of each row of W with x, plus that class's bias
    z = mat_vec(W, x)
    for c in range(len(z)):
        z[c] += b[c]
    return z

def softmax(z):
    exps = [math.exp(value) for value in z]
    total = sum(exps)
    return [e / total for e in exps]

def argmax(values):
    # the position of the largest value
    best = 0
    for c in range(len(values)):
        if values[c] > values[best]:
            best = c
    return best

def loss_one(q, c):
    # minus the log of the probability given to the correct class c
    return -math.log2(q[c])

def average_loss(W, b, X, labels):
    losses = [loss_one(softmax(scores(W, b, X[k])), labels[k]) for k in range(len(labels))]
    return sum(losses) / len(losses)

def transpose(matrix):
    # the rows of the matrix turned into columns
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

def zeros_like(value):
    # a list or matrix of 0s with the same shape as value
    if isinstance(value[0], list):
        return [[0] * len(row) for row in value]
    else:
        return [0] * len(value)

class Parameter:
    """Numbers that training changes: their values, and their gradients."""
    def __init__(self, value):
        self.value = value
        self.grad = zeros_like(value)

class Module:
    """The parent class of every layer and every model."""
    def __init__(self):
        pass

    def __call__(self, X):
        return self.forward(X)

    def parameters(self):
        params = []
        for value in vars(self).values():
            if isinstance(value, Parameter):
                params.append(value)
            elif isinstance(value, Module):
                params.extend(value.parameters())
        return params

class Linear(Module):
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        super().__init__()
        self.weight = Parameter([[random.random() for j in range(n_in)] for c in range(n_out)])
        self.bias = Parameter([random.random() for c in range(n_out)])

    def forward(self, X):
        self._X = X
        out = []
        for x in X:
            z = mat_vec(self.weight.value, x)
            for c in range(len(z)):
                z[c] += self.bias.value[c]
            out.append(z)
        return out

    def backward(self, grad_out):
        W = self.weight.value
        for k in range(len(self._X)):
            for c in range(len(W)):
                for j in range(len(W[c])):
                    self.weight.grad[c][j] += grad_out[k][c] * self._X[k][j]
                self.bias.grad[c] += grad_out[k][c]
        WT = transpose(W)
        return [mat_vec(WT, grad_out[k]) for k in range(len(self._X))]

class SGD:
    """Gradient descent: new value = old value - learning rate * gradient."""
    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = zeros_like(p.value)

    def step(self):
        for p in self.parameters:
            if isinstance(p.value[0], list):
                for i in range(len(p.value)):
                    for j in range(len(p.value[i])):
                        p.value[i][j] -= self.lr * p.grad[i][j]
            else:
                for i in range(len(p.value)):
                    p.value[i] -= self.lr * p.grad[i]

In [ ]:
# the simulated three-class data
random.seed(10)
X = []          # one row [x1, x2] per text: the log counts of positive and of negative words
labels = []     # the class of each text: 0 = positive, 1 = negative, 2 = neutral
for i in range(20):                    # positive texts: many positive words, few negative ones
    X.append([random.gauss(3, 0.5), random.gauss(1, 0.5)])
    labels.append(0)
for i in range(20):                    # negative texts: the reverse
    X.append([random.gauss(1, 0.5), random.gauss(3, 0.5)])
    labels.append(1)
for i in range(10):                    # neutral texts: few of both ...
    X.append([random.gauss(1, 0.5), random.gauss(1, 0.5)])
    labels.append(2)
for i in range(10):                    # ... or many of both
    X.append([random.gauss(3, 0.5), random.gauss(3, 0.5)])
    labels.append(2)
class_names = ["positive", "negative", "neutral"]
num_samples = len(labels)
num_classes = 3

def plot_data():
    markers = ["o", "*", "x"]
    for c in range(num_classes):
        xs = [X[k][0] for k in range(num_samples) if labels[k] == c]
        ys = [X[k][1] for k in range(num_samples) if labels[k] == c]
        plt.scatter(xs, ys, marker=markers[c], s=60, label=class_names[c])
    plt.xlabel("x1: log count of positive words")
    plt.ylabel("x2: log count of negative words")
    plt.legend()

Y = [one_hot(c, num_classes) for c in labels]
print(len(X), "texts;", [labels.count(c) for c in range(num_classes)], "in each class")

---
# Part 1 · The errors

Let's start with a single text and a model that isn't very good yet. (This is the same example as in the lecture, so you can check your numbers against the slides.) The text has the features `x`, and its correct class is **0**, so its one-hot vector is `y = [1, 0, 0]`.

In [ ]:
x = [2.33, 3.72]
y = [1, 0, 0]               # the correct class is 0

W = [[0.77, 0.02],          # class 0's weights
     [0.63, 0.74],          # class 1's weights
     [0.50, 0.22]]          # class 2's weights
b = [0.20, 0.76, 0.17]

q = softmax(scores(W, b, x))
print([round(value, 4) for value in q])         # [0.0488, 0.898, 0.0531]

The model gives the correct class a probability of only 0.05, and gives class 1 almost all the probability. The weights need to change.

As in logistic regression (Week 5, Part 6), we start from the **error** of each probability: the predicted probability minus the correct value. With several classes there is one error **per class**:

`error[c] = q[c] - y[c]`

* the correct class should have probability 1, so if it was given less, its error is **negative**: its probability was too low
* every other class should have probability 0, so its error is **positive**: its probability was too high

### Activity 1 — The errors for one text

Complete the loop so that `error` holds `q[c] - y[c]` for each class `c`.

In [ ]:
error = []
for c in range(num_classes):
    error.append(???)
print([round(value, 4) for value in error])          # you should get [-0.9512, 0.898, 0.0531]

In [ ]:
# ✅ Check
check("error", lambda: [round(v, 4) for v in error] == [-0.9512, 0.898, 0.0531])
check("the errors add up to 0", lambda: abs(sum(error)) < 1e-12)

Why do the errors always add up to 0? *Hint:* what do the probabilities add up to, and what does `y` add up to?

---
# Part 2 · The gradients for one text

The gradients tell us which way to change each weight and bias, and by how much. The rule is exactly the same as for logistic regression, applied to each class in turn:

* the gradient for **class `c`'s weight for feature `j`**, `W[c][j]`, is `error[c] * x[j]`
* the gradient for **class `c`'s bias**, `b[c]`, is `error[c]`

So if a class was given too **much** probability (a positive error), its weights go **down**; if it was given too **little** (a negative error), they go **up**. In each case, the weights for bigger features change more.

The gradients for the weights form a matrix `dW` with the same shape as `W`: one row per class and one column per feature.

| | feature 0 | feature 1 |
|---|---|---|
| class 0 | `error[0] * x[0]` | `error[0] * x[1]` |
| class 1 | `error[1] * x[0]` | `error[1] * x[1]` |
| class 2 | `error[2] * x[0]` | `error[2] * x[1]` |

### Activity 2 — One row of the gradient matrix

First, just class 0's row. Complete the loop so that `row_0` holds `error[0] * x[j]` for each feature `j`.

In [ ]:
row_0 = []
for j in range(len(x)):
    row_0.append(???)
print([round(value, 4) for value in row_0])          # you should get [-2.2163, -3.5384]

In [ ]:
# ✅ Check
check("row_0", lambda: [round(v, 4) for v in row_0] == [-2.2163, -3.5384])

### Activity 3 — The whole gradient matrix

Now every row. The outer loop goes through the classes; for each class `c`, the inner loop makes that class's row, as in Activity 2 but with `error[c]`, and then adds it to `dW`.

1. complete the inner loop
2. `db`: the gradients for the biases. *Hint:* the gradient for `b[c]` is just `error[c]`, so `db` is a copy of `error`: `list(error)` makes one.

In [ ]:
dW = []
for c in range(num_classes):
    row = []
    for j in range(len(x)):
        row.append(???)          # 1.
    dW.append(row)
db = ???                             # 2.

for row in dW:
    print([round(value, 4) for value in row])
# you should get:
# [-2.2163, -3.5384]
# [2.0924, 3.3407]
# [0.1238, 0.1977]

In [ ]:
# ✅ Check
check("dW", lambda: [[round(v, 4) for v in row] for row in dW] == [[-2.2163, -3.5384], [2.0924, 3.3407], [0.1238, 0.1977]])
check("db", lambda: [round(v, 4) for v in db] == [-0.9512, 0.898, 0.0531] and db is not error)

---
# Part 3 · One improvement

Now we update every weight and bias exactly as in Weeks 4, 5 and 7: **new value = old value − learning rate × gradient**.

### Activity 4 — One step of gradient descent

With a learning rate of 0.2:

1. `new_W`: complete the inner loop so that it makes each row of the new weights
2. `new_b`: the new biases (a list comprehension over the classes)
3. `new_q`: the new probabilities for the text

🔮 **Predict** before you run it: should the probability of the correct class, class 0, go up or down? By a lot or a little?

In [ ]:
lr = 0.2

new_W = []
for c in range(num_classes):
    row = []
    for j in range(len(x)):
        row.append(???)       # 1.
    new_W.append(row)
new_b = ???    # 2.

new_q = ???           # 3.
for row in new_W:
    print([round(value, 4) for value in row])
print([round(value, 4) for value in new_b])
print("probabilities before:", [round(value, 4) for value in q])
print("probabilities after: ", [round(value, 4) for value in new_q])
# you should get new weights [1.2133, 0.7277], [0.2115, 0.0719], [0.4752, 0.1805]
# and new biases [0.3902, 0.5804, 0.1594]

In [ ]:
# ✅ Check
check("new_W", lambda: [[round(v, 4) for v in row] for row in new_W] == [[1.2133, 0.7277], [0.2115, 0.0719], [0.4752, 0.1805]])
check("new_b", lambda: [round(v, 4) for v in new_b] == [0.3902, 0.5804, 0.1594])
check("the correct class's probability went up", lambda: new_q[0] > q[0])

One update has moved most of the probability onto the correct class. Training repeats this, over and over, for all the texts.

---
# Part 4 · All the texts at once

For one text, the gradient for `W[c][j]` was `error[c] * x[j]`. For the whole data set we do the same as in Week 5: calculate the gradients for **every** text, and take their **average**. In practice, we add up each text's gradients as we go, and divide by the number of texts at the end.

### Activity 5 — A `gradients` function

Complete `gradients(W, b, X, Y)`, which returns the average gradients `dW` and `db` over all the texts. It starts with `dW` and `db` full of 0s, then, for each text `k`:

1. calculates its probabilities `q`, with `scores` and `softmax`
2. calculates its errors, `q[c] - Y[k][c]` for each class
3. adds its gradients: `error[c] * X[k][j]` to `dW[c][j]`, and `error[c]` to `db[c]`

Finally it divides everything by the number of texts. A function can return several values, separated by commas (LELA60341 Week 4).

In [ ]:
def gradients(W, b, X, Y):
    n_texts = len(X)
    n_classes = len(W)
    n_features = len(W[0])
    dW = [[0] * n_features for c in range(n_classes)]      # a matrix of 0s with the same shape as W
    db = [0] * n_classes
    for k in range(n_texts):
        q = ???                                  # 1.
        error = ???               # 2.
        for c in range(n_classes):
            for j in range(n_features):
                dW[c][j] += ???                           # 3.
            db[c] += ???
    # divide by the number of texts, to get the averages
    for c in range(n_classes):
        for j in range(n_features):
            dW[c][j] = dW[c][j] / n_texts
        db[c] = db[c] / n_texts
    return dW, db

# with one text, the gradients are the ones from Part 2
dW_one, db_one = gradients(W, b, [x], [y])
print([[round(v, 4) for v in row] for row in dW_one], [round(v, 4) for v in db_one])

In [ ]:
# ✅ Check
check("one text: the same as Part 2", lambda: [[round(v, 4) for v in row] for row in dW_one] == [[-2.2163, -3.5384], [2.0924, 3.3407], [0.1238, 0.1977]])
check("one text: the biases", lambda: [round(v, 4) for v in db_one] == [-0.9512, 0.898, 0.0531])
def _two():
    dW2, db2 = gradients(W, b, [x, x], [y, y])
    return all(abs(dW2[c][j] - dW_one[c][j]) < 1e-12 for c in range(3) for j in range(2))
check("the same text twice: the same average", _two)
def _two_texts():
    dW2, db2 = gradients(W, b, [x, [1.0, 0.5]], [y, [0, 0, 1]])
    dWa, dba = gradients(W, b, [x], [y])
    dWb, dbb = gradients(W, b, [[1.0, 0.5]], [[0, 0, 1]])
    return all(abs(dW2[c][j] - (dWa[c][j] + dWb[c][j]) / 2) < 1e-12 for c in range(3) for j in range(2))
check("two different texts: the average of their gradients", _two_texts)

---
# Part 5 · Training

We now have everything we need to train a softmax classifier. Rather than writing a new training loop with `W` and `b`, we use the parts from the LELA60341 Week 8 workbook, which do exactly the same calculations:

* the model is a `Linear` layer with 2 inputs and 3 outputs (last week, Part 7)
* its `backward` method calculates the gradients for all the weights and biases from the gradients of the scores, just like your `gradients` function: `error[c] * x[j]` for each weight, added up over the texts
* `SGD` updates the parameters: **new value = old value − learning rate × gradient**

The one part still missing is the gradient of the loss for each score. For each text `k` and each class `c`, it is the error from Part 1, divided by the number of texts (because the loss is an average):

`(q[c] - y[c]) / n`

### Activity 6 — The gradients of the loss

Complete the method `grad` of `CrossEntropyLoss` (the rest of the class is last week's). For each text it makes the one-hot vector `y` of the correct class, then a row with `(q[c] - y[c]) / n` for each class, where `q` is the text's probabilities, kept in `self._Q[k]`.

The test underneath uses the example from Part 1 to check that `Linear`'s `backward` method, given these gradients, gives exactly your gradients from Part 2.

In [ ]:
class CrossEntropyLoss:
    """The cross-entropy loss for several classes, from the scores."""
    def __call__(self, scores, labels):
        self._Q = [softmax(row) for row in scores]
        self._labels = labels
        losses = [loss_one(self._Q[k], labels[k]) for k in range(len(labels))]
        return sum(losses) / len(losses)

    def grad(self):
        n = len(self._labels)
        G = []
        for k in range(n):
            q = self._Q[k]
            y = one_hot(self._labels[k], len(q))
            G.append(???)
        return G

# the example from Part 1, as a model
example = Linear(2, 3)
example.weight.value = [[0.77, 0.02], [0.63, 0.74], [0.50, 0.22]]
example.bias.value = [0.20, 0.76, 0.17]
loss_fn = CrossEntropyLoss()
loss_fn(example([[2.33, 3.72]]), [0])
print([round(v, 4) for v in loss_fn.grad()[0]])                       # the errors from Part 1: [-0.9512, 0.898, 0.0531]
example.backward(loss_fn.grad())
print([[round(v, 4) for v in row] for row in example.weight.grad])    # your dW from Part 2

In [ ]:
# ✅ Check
check("grad for one text: the errors", lambda: [round(v, 4) for v in loss_fn.grad()[0]] == [-0.9512, 0.898, 0.0531])
check("Linear.backward gives your gradients", lambda: [[round(v, 4) for v in row] for row in example.weight.grad] == [[-2.2163, -3.5384], [2.0924, 3.3407], [0.1238, 0.1977]])
def _all():
    random.seed(3)
    m = Linear(2, 3)
    f = CrossEntropyLoss()
    f(m(X), labels)
    m.backward(f.grad())
    dW, db = gradients(m.weight.value, m.bias.value, X, Y)
    return all(abs(m.weight.grad[c][j] - dW[c][j]) < 1e-12 for c in range(3) for j in range(2)) and all(abs(m.bias.grad[c] - db[c]) < 1e-12 for c in range(3))
check("for all 60 texts: the same as your gradients function", _all)

### Activity 7 — The training loop

Now the training loop: the same five-line loop as in the LELA60341 Week 8 workbook. Each pass through the data (each **epoch**):

1. forgets the old gradients
2. calculates the scores of every text (the forward pass)
3. calculates the loss, and records it
4. calculates the gradients (the backward pass): `model.backward(loss_fn.grad())`
5. updates the parameters

The model starts with random weights and biases (`random.seed(10)` first), and the learning rate is 0.5. Fill in the blanks. Training takes a few seconds.

In [ ]:
random.seed(10)
model = Linear(2, 3)
loss_fn = CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=0.5)
n_epochs = 1000
loss_history = []

for epoch in range(n_epochs):
    ???                                         # 1.
    all_scores = ???                                         # 2.
    loss = ???                            # 3.
    loss_history.append(loss)
    ???                                # 4.
    ???                                              # 5.

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print("final loss:", round(loss_history[-1], 3))       # you should get 0.233
for row in model.weight.value:
    print([round(value, 2) for value in row])
print([round(value, 2) for value in model.bias.value])

In [ ]:
# ✅ Check
check("the loss went down", lambda: loss_history[-1] < loss_history[0])
check("final loss about 0.233", lambda: abs(loss_history[-1] - 0.233) < 0.001)
check("positive words push the positive class up", lambda: model.weight.value[0][0] > model.weight.value[0][1])
check("negative words push the negative class up", lambda: model.weight.value[1][1] > model.weight.value[1][0])

Look at the trained weights. Do they make sense? Compare them with the weights we used last week.

---
# Part 6 · Using the trained model

### Activity 8 — Predictions and accuracy

1. `predictions`: the predicted class of every text, with the trained `model`. *Hint:* `argmax` of each row of the model's scores.
2. `accuracy`: the proportion of texts whose prediction equals their label

In [ ]:
predictions = ???
accuracy = ???
print("accuracy:", accuracy)        # you should get 0.95
wrong = [k for k in range(num_samples) if predictions[k] != labels[k]]
print("misclassified texts:", wrong)

In [ ]:
# ✅ Check
check("predictions", lambda: len(predictions) == 60 and predictions[0] == argmax(model([X[0]])[0]))
check("accuracy", lambda: abs(accuracy - 0.95) < 1e-9)

Here is the plot again, with each misclassified text circled in red. Which texts does the model get wrong? A softmax classifier divides the space into regions with **straight** borders. That works quite well here, because the neutral texts lie in a diagonal band between the positive and negative ones.

In [ ]:
plot_data()
plt.scatter([X[k][0] for k in wrong], [X[k][1] for k in wrong], s=250, facecolors="none", edgecolors="red")
plt.show()

### Activity 9 — Classifying new texts

Complete `predict_sentiment(n_positive, n_negative)` from last week, this time with the model you have trained. Remember that the features are the **logs** of the counts, and that a model always takes a **list** of examples and returns a list of rows.

In [ ]:
def predict_sentiment(n_positive, n_negative):
    x = ???
    return ???

print(predict_sentiment(20, 2), predict_sentiment(2, 20), predict_sentiment(3, 3), predict_sentiment(20, 20))
# you should get positive negative neutral neutral

In [ ]:
# ✅ Check
check("20 positive, 2 negative", lambda: predict_sentiment(20, 2) == "positive")
check("2 positive, 20 negative", lambda: predict_sentiment(2, 20) == "negative")
check("3 positive, 3 negative", lambda: predict_sentiment(3, 3) == "neutral")

---
# Part 7 · Precision and recall for several classes

In Week 7 we calculated precision and recall for a two-class classifier, treating "positive" as the class of interest. With several classes we do the same for **each class in turn**, treating that class as the class of interest and all the other classes together as "not it":

* **true positives** for class `c`: texts predicted as `c` that really are `c`
* **false positives** for class `c`: texts predicted as `c` that are really something else
* **false negatives** for class `c`: texts that really are `c` but were predicted as something else

This gives a precision and a recall for **each** class.

### Activity 10 — Precision and recall for one class

Complete `precision_recall(predicted, actual, c)`, which returns the precision and the recall for class `c`. The condition for true positives is done for you.

In [ ]:
def precision_recall(predicted, actual, c):
    tp = 0
    fp = 0
    fn = 0
    for k in range(len(predicted)):
        if predicted[k] == c and actual[k] == c:
            tp += 1
        if ???:      # predicted c, but really something else
            fp += 1
        if ???:      # really c, but predicted something else
            fn += 1
    return ???, ???

# a small example
print(precision_recall([0, 1, 1, 2, 2, 0], [0, 1, 2, 2, 1, 1], 1))      # you should get (0.5, 0.3333333333333333)

for c in range(num_classes):
    p, r = precision_recall(predictions, labels, c)
    print(class_names[c], " precision:", round(p, 3), " recall:", round(r, 3))

In [ ]:
# ✅ Check
check("the small example, class 0", lambda: precision_recall([0, 1, 1, 2, 2, 0], [0, 1, 2, 2, 1, 1], 0) == (0.5, 1.0))
check("the small example, class 1", lambda: precision_recall([0, 1, 1, 2, 2, 0], [0, 1, 2, 2, 1, 1], 1) == (0.5, 1 / 3))
check("the small example, class 2", lambda: precision_recall([0, 1, 1, 2, 2, 0], [0, 1, 2, 2, 1, 1], 2) == (0.5, 0.5))

Which class has the lowest precision or recall? Does that fit with the plot in Part 6?

### Activity 11 — Macro-averaged precision and recall

It is useful to have a single number for the whole classifier. **Macro-averaging** takes the plain average over the classes, so that every class counts equally, however many texts it has.

Complete `macro_precision_recall(predicted, actual, n_classes)`: for each class, add its precision and recall to the lists, then return the two averages.

In [ ]:
def macro_precision_recall(predicted, actual, n_classes):
    precisions = []
    recalls = []
    for c in range(n_classes):
        p, r = ???
        precisions.append(p)
        recalls.append(r)
    return ???, ???

print(macro_precision_recall([0, 1, 1, 2, 2, 0], [0, 1, 2, 2, 1, 1], 3))     # you should get (0.5, 0.611...)
macro_p, macro_r = macro_precision_recall(predictions, labels, num_classes)
print("macro precision:", round(macro_p, 3), " macro recall:", round(macro_r, 3))

In [ ]:
# ✅ Check
check("the small example", lambda: macro_precision_recall([0, 1, 1, 2, 2, 0], [0, 1, 2, 2, 1, 1], 3) == (0.5, (1.0 + 1 / 3 + 0.5) / 3))
check("the trained model", lambda: (macro_p, macro_r) == macro_precision_recall(predictions, labels, 3))

---
## ✏️ Summary

You have trained and evaluated a softmax classifier:

1. the **errors** for a text are `q[c] - y[c]` for each class: negative for the correct class, positive for the others
2. the **gradient** for `W[c][j]` is `error[c] * x[j]`, and for `b[c]` it is `error[c]`, averaged over the texts: exactly the rule for logistic regression, once per class
3. **training** repeats the update **new value = old value − learning rate × gradient** many times. With the parts from the LELA60341 Week 8 workbook, a softmax classifier is a `Linear` layer with one output per class, trained with `CrossEntropyLoss`, `SGD` and the usual five-line loop
4. the **predicted class** is the one with the highest probability
5. **precision and recall** can be calculated for each class in turn, and **macro-averaged** over the classes